## This file deals with the following Optimizations done by Delta Lake in Databricks

1. OPTIMIZE
2. ZORDER
3. VACCUUM

In [0]:
from pyspark.sql import Row

# Create initial DataFrame with sample data
data = [Row(id=1, name='Alice', age=30), Row(id=2, name='Bob', age=25), Row(id=3, name='Charlie', age=35)]
df = spark.createDataFrame(data)

# Overwrite and create a managed Delta table with the DataFrame in the training.delta_demo schema
df.write.format('delta').mode('overwrite').saveAsTable('delta_table_practice.delta.people_delta_demo_2')

# Read the Delta table to verify creation and display its contents
delta_df = spark.read.table('delta_table_practice.delta.people_delta_demo_2')
display(delta_df)


we are writing it to a volume so that we can see the files being stored to better understand the operations happening because in free version we get 'default storage' when we store a table which means we can't see how the folders are made which are storing the delta table

In [0]:
df.write.format('delta').mode('overwrite').save('/Volumes/delta_table_practice/delta/delta_volume/delta_test_table_2')

In [0]:
df.write.format('delta').mode('append').save('/Volumes/delta_table_practice/delta/delta_volume/delta_test_table_2')


spark is very efficient when file size=128 MB. Since we have a large number of small files, we can combine them all to a larger file using 'Optimize' command.

In [0]:
%sql
optimize '/Volumes/delta_table_practice/delta/delta_volume/delta_test_table_2'

In [0]:
%sql
optimize delta_table_practice.delta.people_delta_demo_2

%md
## OPTIMIZE and ZORDER in Delta Lake

### OPTIMIZE
- Compacts many **small files** into fewer, **larger files** (target ~1 GB by default).
- Fixes the **small file problem**, which speeds up reads and reduces metadata overhead.
- Does not change the data, only how it is laid out in files.
- Old files remain until `VACUUM` is run.

```sql
OPTIMIZE my_table;
```

### ZORDER BY
- An optional clause of `OPTIMIZE` that **sorts and co-locates rows** with similar values of the chosen column(s) in the same files.
- Each file ends up covering a **narrow range** of the column, so Delta can **skip files** that don't match a query's filter (**data skipping**).
- Best for **high-cardinality** columns that are frequently used in `WHERE` filters or joins (e.g. `id`, timestamps).
- With one column it behaves like a sort. With multiple columns it uses a **Z-order curve** to cluster on all of them at once.
- Keep the column list short (**1 to 3 columns**). Partition columns can't be Z-ordered.

```sql
OPTIMIZE my_table
ZORDER BY (id);
```

### Key difference
| | OPTIMIZE | ZORDER BY |
|---|---|---|
| Controls | File **size** | File **contents** (row arrangement) |
| Benefit | Fewer, larger files | Faster filtered queries via data skipping |

### Good to know
- Both rewrite data, so they cost compute. Run them **periodically**, not after every write.
- On newer Databricks runtimes, **Liquid Clustering** (`CLUSTER BY`) is the recommended replacement for partitioning and Z-ORDER.

In [0]:
%sql
optimize delta_table_practice.delta.people_delta_demo_2 zorder by (id)

## VACUUM in Delta Lake

### What it does
- **Permanently deletes** data files that are no longer referenced by the Delta table and are **older than the retention period**.
- Frees up **storage**. `OPTIMIZE` and `UPDATE`/`DELETE`/`MERGE` only mark old files as removed, they don't physically delete them.
- Does not change the current table data, only cleans up old files.

### Retention (defaults)
- `delta.deletedFileRetentionDuration`: **7 days**, the minimum age of removed files before `VACUUM` can delete them.
- `delta.logRetentionDuration`: **30 days**, how long transaction log history is kept (limits time travel).
- Both are **table properties**, set with `ALTER TABLE ... SET TBLPROPERTIES`, not with `SET`.

```sql
ALTER TABLE my_table SET TBLPROPERTIES (
  'delta.logRetentionDuration' = 'interval 30 days',
  'delta.deletedFileRetentionDuration' = 'interval 7 days'
);
```

### Syntax
```sql
-- preview only, deletes nothing (lists up to 1,000 files)
VACUUM my_table DRY RUN;

-- actually delete
VACUUM my_table;

-- explicit retention (168 hours = 7 days)
VACUUM my_table RETAIN 168 HOURS;
```

### Retention shorter than 7 days
- Blocked by default by a safety check.
- Can be disabled for the **current session only** (practice or demo use only):

```sql
SET spark.databricks.delta.retentionDurationCheck.enabled = false;
VACUUM my_table RETAIN 0 HOURS DRY RUN;   -- always preview first
VACUUM my_table RETAIN 0 HOURS;
SET spark.databricks.delta.retentionDurationCheck.enabled = true;   -- re-enable
```

### Risks
- **Time travel breaks** for versions that depended on the deleted files.
- **Concurrent readers/streams can fail** if they still read vacuumed files.
- **Not recoverable**, since `RESTORE` can't bring back deleted files.
- Avoid short retention on production tables.

### Typical pattern
- Run `OPTIMIZE` first (compacts files and marks the small ones as removed).
- Run `VACUUM` later to physically delete them once the retention period has passed.
- A `DRY RUN` right after creating or modifying a table often returns **no rows**, because nothing is old enough yet.

`delta.logRetentionDuration`: 30 days (how long transaction log history is kept, which limits time travel).
`delta.deletedFileRetentionDuration`: 7 days (how long removed data files are kept before VACUUM can delete them).

In [0]:
%sql
ALTER TABLE delta_table_practice.delta.people_delta_demo_2
SET TBLPROPERTIES (
  'delta.logRetentionDuration' = 'interval 30 days',
  'delta.deletedFileRetentionDuration' = 'interval 7 days'
);

In [0]:
%sql
vacuum delta_table_practice.delta.people_delta_demo_2 dry run 

In [0]:
%sql
SET spark.databricks.delta.retentionDurationCheck.enabled = false;

VACUUM delta_table_practice.delta.people_delta_demo_2 RETAIN 0 HOURS DRY RUN;   -- preview first
VACUUM delta_table_practice.delta.people_delta_demo_2 RETAIN 0 HOURS;           -- actually delete